# EDA — Predicting Electric Vehicle Purchases

**Competition:** Kaggle Playground Series S6E9  
**Metric:** ROC-AUC  
**Task:** Predict whether a person will buy an electric vehicle (`Will_Buy_EV`: Yes / No)

This analysis was conducted independently as part of a Kaggle competition.  
Notebook structure and comments were reviewed and edited with Claude Sonnet 4.6.

---

## Structure
1. Load Data
2. Dataset Overview
3. Target Analysis
4. Strong Features
5. Weak Features
6. How `Range_Anxiety_Level` Is Generated
7. Key Finding: The Hidden Purchase Formula
8. EDA Summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

# Clean, minimal chart style — no top/right borders, subtle grid
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('whitegrid')

print('Libraries loaded')

## 1. Load Data

In [ ]:
BASE_DIR = Path('data') / 'raw'

train = pd.read_csv(BASE_DIR / 'train.csv')
test  = pd.read_csv(BASE_DIR / 'test.csv')

# Work on a lowercase copy to keep column names consistent
df = train.copy(deep=True)
df.columns = df.columns.str.lower()

print(f'Train : {train.shape[0]:,} rows x {train.shape[1]} columns')
print(f'Test  : {test.shape[0]:,} rows x {test.shape[1]} columns')

## 2. Dataset Overview

In [ ]:
# Quick diagnostics: shape, duplicates, missing values
print('=' * 50)
print(f'  Rows       : {df.shape[0]:,}')
print(f'  Columns    : {df.shape[1]}')
print(f'  Duplicates : {df.duplicated().sum()}')
print(f'  Nulls      : {df.isna().sum().sum()}')
print(f'  Memory     : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print('=' * 50)

pd.DataFrame({
    'dtype'   : df.dtypes,
    'non-null': df.count(),
    'unique'  : df.nunique(),
}).style.set_caption('Column Summary')

In [ ]:
# Sample rows to understand the data structure
df.sample(5, random_state=42)

**Observations:**
- No missing values and no duplicates — the dataset is clean.
- 7 numeric features, 6 categorical features, 1 binary target.
- The target column `will_buy_ev` is a string ("Yes" / "No") and needs to be encoded.

## 3. Target Analysis

In [ ]:
# Binary target for numeric calculations
df['will_buy_ev_bin'] = (df['will_buy_ev'] == 'Yes').astype(int)

# Helper segments — used in later plots
df['income_segment'] = pd.cut(
    df['annual_income_usd'],
    bins=[30000, 60000, 90000, 130000, 189000],
    labels=['low', 'mid', 'mid-high', 'high'],
    include_lowest=True
)
df['commute_segment'] = pd.cut(
    df['daily_commute_km'],
    bins=[5, 15, 30, 60, 98.7],
    labels=['5-15 km', '15-30 km', '30-60 km', '60-99 km'],
    include_lowest=True
)

counts = df['will_buy_ev'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#e07070', '#5b8fe0']
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3000,
            f'{val:,}\n({val/len(df):.1%})', ha='center', fontsize=11)

ax.set_title('Target Class Distribution', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout()
plt.show()

print(f'Class ratio  No : Yes = {counts["No"]:,} : {counts["Yes"]:,}  (~{counts["No"]/counts["Yes"]:.0f}:1)')

**Observation:** The dataset is imbalanced — only **17.5%** of people buy an EV.  
This is typical for high-cost purchase prediction tasks. The model must learn to identify the minority class.

## 4. Strong Features

A feature is "strong" if the EV purchase rate changes significantly across its values.  
The wider the spread — the more useful the feature is for the model.

In [ ]:
# --- Environmental Concern Level ---
# A self-reported score from 1 (not concerned) to 5 (very concerned).
# This is the single most predictive feature: purchase rate ranges from 0.6% to 51.8%.

concern_rate = df.groupby('environmental_concern_level')['will_buy_ev_bin'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(concern_rate.index.astype(str), concern_rate.values * 100,
              color='#5b8fe0', edgecolor='white', width=0.55)
for bar, val in zip(bars, concern_rate.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1%}', ha='center', fontsize=10, fontweight='bold')

ax.set_title('Environmental Concern Level — Purchase Rate by Score', fontsize=12, fontweight='bold')
ax.set_xlabel('Concern Level (1 = low, 5 = high)')
ax.set_ylabel('EV Purchase Rate (%)')
plt.tight_layout()
plt.show()

print('EV purchase rate by environmental concern level:')
print(concern_rate.map('{:.1%}'.format))

In [ ]:
# --- Range Anxiety Level ---
# How worried is a person about the battery running out before reaching a charger?
# Low anxiety → 18.9% buy EV. High anxiety → nearly nobody buys (0.1%).

anxiety_rate = df.groupby('range_anxiety_level')['will_buy_ev_bin'].mean().reindex(['Low', 'Medium', 'High'])

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#70c07a', '#f0c060', '#e07070']
bars = ax.bar(anxiety_rate.index, anxiety_rate.values * 100,
              color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, anxiety_rate.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')

ax.set_title('Range Anxiety Level — Purchase Rate', fontsize=12, fontweight='bold')
ax.set_xlabel('Range Anxiety Level')
ax.set_ylabel('EV Purchase Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Subsidy Available ---
# Is a government EV subsidy available to this person?
# Without subsidy: 0.6% buy. With subsidy: 27.5% buy.
# The strongest binary feature in the dataset.

subsidy_rate = df.groupby('subsidy_available')['will_buy_ev_bin'].mean()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(subsidy_rate.index, subsidy_rate.values * 100,
              color=['#e07070', '#70c07a'], edgecolor='white', width=0.4)
for bar, val in zip(bars, subsidy_rate.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')

ax.set_title('Subsidy Available — Purchase Rate', fontsize=12, fontweight='bold')
ax.set_xlabel('Subsidy Available')
ax.set_ylabel('EV Purchase Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Annual Income ---
# Higher income correlates with higher EV purchase rate.
# However, the spread (6.4% → 35.1%) is moderate compared to concern or anxiety.

income_rate = df.groupby('income_segment', observed=True)['will_buy_ev_bin'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(income_rate.index.astype(str), income_rate.values * 100,
              color='#9b8fd4', edgecolor='white', width=0.55)
for bar, val in zip(bars, income_rate.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1%}', ha='center', fontsize=10, fontweight='bold')

ax.set_title('Annual Income Segment — Purchase Rate', fontsize=12, fontweight='bold')
ax.set_xlabel('Income Segment')
ax.set_ylabel('EV Purchase Rate (%)')
plt.tight_layout()
plt.show()

**Summary — Strong Features:**

| Feature | Purchase Rate Range | Signal Strength |
|---|---|---|
| `environmental_concern_level` | 0.6% → 51.8% | Very strong |
| `range_anxiety_level` | 0.1% → 18.9% | Very strong |
| `subsidy_available` | 0.6% → 27.5% | Very strong |
| `annual_income_usd` | 6.4% → 35.1% | Moderate |

These four features are the primary drivers of EV purchase behavior in this dataset.

## 5. Weak Features

These features show little direct effect on the purchase rate.  
However, some of them act **indirectly** — by influencing `range_anxiety_level` (see Section 6).

In [ ]:
# Features with a small but visible signal
weak = {
    'home_charging_possible': df.groupby('home_charging_possible')['will_buy_ev_bin'].mean(),
    'city_type':              df.groupby('city_type')['will_buy_ev_bin'].mean(),
    'commute_segment':        df.groupby('commute_segment', observed=True)['will_buy_ev_bin'].mean(),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, series) in zip(axes, weak.items()):
    bars = ax.bar(series.index.astype(str), series.values * 100,
                  color='#b0c4de', edgecolor='white', width=0.5)
    for bar, val in zip(bars, series.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                f'{val:.1%}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.set_ylabel('EV Purchase Rate (%)')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Weak Features — Small Direct Signal', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Features with no meaningful direct signal (spread < 4 percentage points)
no_signal = pd.DataFrame({
    'Feature': [
        'age', 'gender', 'current_car_type',
        'number_of_cars_owned',
        'charging_stations_near_home',
        'charging_stations_near_work'
    ],
    'Rate Min': ['17.4%', '17.0%', '15.6%', '16.6%', '16.2%', '16.4%'],
    'Rate Max': ['17.8%', '17.3%', '18.1%', '17.8%', '19.8%', '19.1%'],
    'Spread'  : ['0.4 pp', '0.3 pp', '2.5 pp', '1.2 pp', '3.6 pp', '2.7 pp'],
    'Verdict' : ['No signal', 'No signal', 'Very weak', 'Very weak', 'Indirect', 'Indirect'],
})
print('Features with no meaningful direct signal:')
no_signal

**Note:** `charging_stations_near_home` and `charging_stations_near_work` appear weak here,  
but they play an important **indirect** role — they feed into the `worry_score` that determines `range_anxiety_level`.

## 6. How `Range_Anxiety_Level` Is Generated

Since this is a synthetic dataset (stated in the competition description), we can investigate *how* the data was generated.

Community analysis of the original 10,000-row dataset revealed the following formula:

```
worry_score = daily_commute_km
            − 5 × charging_stations_near_home
            − 5 × charging_stations_near_work
            − 150 × (home_charging_possible == "Yes")
```

**Thresholds:**
- `worry_score ≤ −25` → **Low** anxiety  
- `−25 < worry_score ≤ 75` → **Medium** anxiety  
- `worry_score > 75` → **High** anxiety

This explains why `city_type` and `home_charging_possible` have an indirect effect:  
they influence charging availability, which determines `worry_score`, which determines `range_anxiety_level`.

In [ ]:
# Compute worry_score using the discovered formula
df['worry_score'] = (
    df['daily_commute_km']
    - 5 * df['charging_stations_near_home']
    - 5 * df['charging_stations_near_work']
    - 150 * (df['home_charging_possible'] == 'Yes').astype(int)
)

# Plot: worry_score distribution by anxiety level
# Threshold lines should clearly separate the three groups
fig, ax = plt.subplots(figsize=(11, 4))

colors = {'Low': '#70c07a', 'Medium': '#f0c060', 'High': '#e07070'}
for lvl, col in colors.items():
    subset = df[df['range_anxiety_level'] == lvl]['worry_score']
    ax.hist(subset, bins=60, alpha=0.65, color=col, label=lvl, edgecolor='white')

ax.axvline(-25, color='#333', linestyle='--', linewidth=2, label='Threshold: −25')
ax.axvline(75,  color='#333', linestyle=':',  linewidth=2, label='Threshold: +75')
ax.set_xlabel('worry_score')
ax.set_ylabel('Count')
ax.set_title('worry_score Explains Range Anxiety Level', fontsize=12, fontweight='bold')
ax.legend(title='range_anxiety_level')
plt.tight_layout()
plt.show()

# Verify: what fraction of rows match the expected anxiety level?
def expected_anxiety(score):
    if score <= -25:  return 'Low'
    if score <= 75:   return 'Medium'
    return 'High'

df['anxiety_expected'] = df['worry_score'].apply(expected_anxiety)
match_rate = (df['anxiety_expected'] == df['range_anxiety_level']).mean()
print(f'Formula match rate: {match_rate:.1%}  (small gap = random noise in synthetic data)')

In [ ]:
# Why does city_type matter indirectly?
# Urban areas have more charging stations → lower worry_score → less anxiety → more EV purchases.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
city_colors = {'Urban': '#5b8fe0', 'Suburban': '#70c07a', 'Rural': '#f0a060'}

for ax, col in zip(axes, ['charging_stations_near_home', 'charging_stations_near_work']):
    for city, color in city_colors.items():
        subset = df[df['city_type'] == city][col]
        ax.hist(subset, bins=20, alpha=0.6, color=color, label=city, edgecolor='white')
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Number of Charging Stations')
    ax.set_ylabel('Count')
    ax.legend(title='City Type')

plt.suptitle('City Type → Charging Availability → worry_score → Range Anxiety → EV Purchase',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Key Finding: The Hidden Purchase Formula

If `range_anxiety_level` is generated from a formula — could the **target itself** also follow a formula?

Community analysis confirmed it. The purchase decision is governed by:

```
buy_score = 1.2 × (annual_income_usd / 100,000)
          + 0.6 × environmental_concern_level
          + 2.0 × (subsidy_available == "Yes")
          − 1.0 × (range_anxiety_level == "Medium")
          − 3.0 × (range_anxiety_level == "High")
```

**If `buy_score > ~5.5` (plus random noise) → the person buys an EV.**

What the formula tells us:
- Income and concern level drive the score upward.
- A subsidy adds a large bonus (+2.0) — it is the strongest single factor.
- Anxiety reduces the score, with "High" being nearly disqualifying (−3.0).
- Age, gender, city type, car type, commute, and charger counts do **not** appear in the formula — they only act indirectly.

In [ ]:
# Compute buy_score using the discovered formula
df['buy_score'] = (
    1.2 * df['annual_income_usd'] / 1e5
    + 0.6 * df['environmental_concern_level']
    + 2.0 * (df['subsidy_available'] == 'Yes').astype(int)
    + df['range_anxiety_level'].map({'Low': 0, 'Medium': -1, 'High': -3})
)

# Bin the score and compute purchase rate per bin
df['buy_score_bin'] = pd.cut(df['buy_score'], bins=np.arange(0, 11, 0.5))
score_rate = df.groupby('buy_score_bin', observed=True)['will_buy_ev_bin'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
centers = [b.mid for b in score_rate.index]
ax.bar(centers, score_rate.values * 100, width=0.45,
       color='#5b8fe0', edgecolor='white', label='Observed purchase rate')
ax.axvline(5.5, color='#e05050', linestyle='--', linewidth=2, label='Threshold ≈ 5.5')

ax.set_xlabel('buy_score (formula, without noise)')
ax.set_ylabel('EV Purchase Rate (%)')
ax.set_title('buy_score vs. Actual Purchase Rate — Formula Works', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

# How well does the formula predict purchases — without any model training?
auc_buy   = roc_auc_score(df['will_buy_ev_bin'], df['buy_score'])
auc_worry = roc_auc_score(df['will_buy_ev_bin'], -df['worry_score'])

print(f'ROC-AUC  buy_score   (no model, formula only) : {auc_buy:.4f}')
print(f'ROC-AUC  worry_score (no model, formula only) : {auc_worry:.4f}')
print()
print('For reference: our tuned LightGBM baseline     : 0.9418')
print()
print('The formula alone reaches 0.937 ROC-AUC.')
print('This is the ceiling we are trying to approach with feature engineering.')

In [ ]:
# Average contribution of each component to the final buy_score
components = pd.DataFrame({
    'Income  (×1.2 / 100k)' : 1.2 * df['annual_income_usd'] / 1e5,
    'Concern (×0.6)'        : 0.6 * df['environmental_concern_level'],
    'Subsidy (×2.0)'        : 2.0 * (df['subsidy_available'] == 'Yes').astype(int),
    'Anxiety (0 / −1 / −3)' : df['range_anxiety_level'].map({'Low': 0, 'Medium': -1, 'High': -3}),
})

fig, ax = plt.subplots(figsize=(9, 4))
means = components.mean()
colors = ['#9b8fd4', '#70c07a', '#5b8fe0', '#e07070']
bars = ax.bar(means.index, means.values, color=colors, edgecolor='white', width=0.5)
ax.axhline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            val + (0.05 if val >= 0 else -0.15),
            f'{val:+.2f}', ha='center', fontsize=10, fontweight='bold')

ax.set_title('Average Contribution of Each Formula Component', fontsize=12, fontweight='bold')
ax.set_ylabel('Average contribution to buy_score')
plt.tight_layout()
plt.show()

## 8. EDA Summary

### Feature Signal Strength and Feature Engineering Decisions

| Feature | Purchase Rate Spread | Signal | Used in Feature Engineering |
|---|---|---|---|
| `environmental_concern_level` | 0.6% → 51.8% | Direct (formula) | → `buy_score` component |
| `subsidy_available` | 0.6% → 27.5% | Direct (formula) | → `buy_score` component; binary encode; `income_x_subsidy`, `concern_x_subsidy` interactions |
| `range_anxiety_level` | 0.1% → 18.9% | Direct (formula) | → `buy_score` component; ordinal encode (Low=0, Medium=1, High=2) |
| `annual_income_usd` | 6.4% → 35.1% | Direct (formula) | → `buy_score` component; `income_x_subsidy`, `income_per_car` interactions; TE key |
| `home_charging_possible` | 12.7% → 19.6% | Indirect (worry_score) | → `worry_score` component; binary encode |
| `daily_commute_km` | 10.7% → 19.1% | Indirect (worry_score) | → `worry_score` component; `commute_per_charger` interaction; TE key |
| `charging_stations_near_home` | 16.2% → 19.8% | Indirect (worry_score) | → `worry_score` component |
| `charging_stations_near_work` | 16.4% → 19.1% | Indirect (worry_score) | → `worry_score` component |
| `city_type` | 16.1% → 19.3% | Indirect (via chargers) | → One-hot encode |
| `current_car_type` | 15.6% → 18.1% | Weak direct | → One-hot encode |
| `home_charging_possible` | — | Indirect | → Binary encode |
| `gender` | 17.0% → 17.3% | No signal | → One-hot encode (kept for completeness) |
| `age` | 17.4% → 17.8% | No signal | → Kept as numeric; `age_x_income` interaction |
| `number_of_cars_owned` | 16.6% → 17.8% | No signal | → `income_per_car` interaction |

### Key Conclusions

1. **Three features dominate:** `environmental_concern_level`, `subsidy_available`, and `range_anxiety_level` together explain most of the variance in the target.

2. **The dataset has a deterministic structure.** Both `range_anxiety_level` and `Will_Buy_EV` are generated by formulas with added noise. Knowing the formulas gives us a strong baseline and a clear ceiling.

3. **Indirect features still matter.** `daily_commute_km`, `charging_stations_*`, and `home_charging_possible` do not predict purchases directly, but they shape `range_anxiety_level`, which is a top predictor.

4. **Feature engineering strategy:** add `buy_score` and `worry_score` as explicit features so the model does not need to rediscover the formula. Use Nested Target Encoding on income and commute buckets to capture residual signal the formula misses.